In [33]:
import pandas as pd  
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, models
from tensorflow.keras import regularizers
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
df = pd.read_csv('Data\poses_dataset.csv')

<>:12: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:12: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\tatir\AppData\Local\Temp\ipykernel_33256\2001294153.py:12: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  df = pd.read_csv('Data\poses_dataset.csv')


In [8]:
X= df.drop(columns=['timestamp', 'video_name', 'label'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

In [23]:
rf_model = RandomForestClassifier(
      n_estimators=200,
      max_depth=10,
      min_samples_split=2,
      min_samples_leaf=2,
      random_state=42,
      class_weight='balanced'
  )

In [25]:
rf_model.fit(X_train, y_train)

print(rf_model.score(X_train, y_train))
print(rf_model.score(X_val, y_val))
print(rf_model.score(X_test, y_test))

0.971895758814512
0.900459418070444
0.8744257274119449


In [26]:
def build_kick_model(input_shape):
      model = models.Sequential([
          # Input Layer -> Hidden Layer 1
          layers.Dense(64, activation='relu', input_shape=(input_shape,), kernel_regularizer=regularizers.l2(0.01)),
          layers.Dropout(0.2),

          # Hidden Layer 2
          layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
          layers.Dropout(0.2),

          # Hidden Layer 3
          layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(0.01)),

          # Output Layer (Sigmoid for binary 0/1 classification)
          layers.Dense(1, activation='sigmoid')
      ])

      # Compile the model
      model.compile(
          optimizer='adam',
          loss='binary_crossentropy',
          metrics=['accuracy']
      )

      return model

In [27]:
early_stop = EarlyStopping(
      monitor='val_loss',
      patience=10,          # Wait 10 epochs before giving up
      restore_best_weights=True # Give me the model from the best epoch, not the last one
  )

In [30]:
nn_model = build_kick_model(X_train.shape[1])

nn_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stop])

print(nn_model.evaluate(X_test, y_test))

Epoch 1/100


c:\Users\tatir\OneDrive\Documentos\Projetos Pessoais\TaekwondoProject\Taekwondo_Kick_Prediction\.venv\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6929 - loss: 1.7613 - val_accuracy: 0.8270 - val_loss: 1.4870
Epoch 2/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8273 - loss: 1.2763 - val_accuracy: 0.8530 - val_loss: 1.1285
Epoch 3/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8682 - loss: 0.9877 - val_accuracy: 0.8576 - val_loss: 0.8982
Epoch 4/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8728 - loss: 0.8233 - val_accuracy: 0.8622 - val_loss: 0.7604
Epoch 5/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8789 - loss: 0.6970 - val_accuracy: 0.8591 - val_loss: 0.7013
Epoch 6/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8861 - loss: 0.6208 - val_accuracy: 0.8729 - val_loss: 0.5756
Epoch 7/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8906 - loss: 0.5445 - val_accuracy: 0.8637 - val_loss: 0.5531
Epoch 8/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8942 - loss: 0.4984 - val_accuracy: 0.8729 - val_loss: 0.

In [28]:
def evaluate_model(name, model, X_test, y_test):
      # For NN, we need to convert probabilities to 0 or 1
      if hasattr(model, 'predict_proba') or 'RandomForest' in str(type(model)):
          preds = model.predict(X_test)
      else: # TensorFlow model
          probs = model.predict(X_test)
          preds = (probs > 0.5).astype(int).flatten()

      print(f"\n--- {name} Report ---")
      print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
      print(classification_report(y_test, preds))

In [31]:
evaluate_model("Random Forest", rf_model, X_test, y_test)
evaluate_model("Neural Network", nn_model, X_test, y_test)


--- Random Forest Report ---
Accuracy: 0.8744
              precision    recall  f1-score   support

           0       0.90      0.84      0.87       320
           1       0.85      0.91      0.88       333

    accuracy                           0.87       653
   macro avg       0.88      0.87      0.87       653
weighted avg       0.88      0.87      0.87       653

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

--- Neural Network Report ---
Accuracy: 0.9005
              precision    recall  f1-score   support

           0       0.92      0.88      0.90       320
           1       0.89      0.92      0.90       333

    accuracy                           0.90       653
   macro avg       0.90      0.90      0.90       653
weighted avg       0.90      0.90      0.90       653



In [34]:
nn_model.save('kick_detection_model.h5')
joblib.dump(scaler, 'scaler.pkl')
print("Model saved as kick_detection_model.h5")

Model saved as kick_detection_model.h5
